# Task 2B — QLoRA fine-tuning of Phi-3-mini (Colab T4)
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nazifmhd/Task-1---Financial-AI/blob/main/task2_genai/02_finetune_qlora.ipynb)

**Before running:** *Runtime → Change runtime type → T4 GPU*. Add the Colab Secret **`HF_TOKEN`**
(a Hugging Face *write* token) to push the merged model. W&B is optional: if `WANDB_API_KEY` is set,
the run is also logged there. Otherwise losses are logged manually in this notebook, which the brief allows.

Student: `microsoft/Phi-3-mini-4k-instruct` (3.8B). Teacher was `openai/gpt-oss-120b` (different family).

In [ ]:
# Colab: clone the repo (data + src) and install pinned dependencies.
import os, sys, subprocess
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    if not os.path.exists("Task-1---Financial-AI"):
        subprocess.run(["git", "clone", "-q", "https://github.com/nazifmhd/Task-1---Financial-AI.git"], check=True)
    os.chdir("Task-1---Financial-AI/task2_genai")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
elif os.path.basename(os.getcwd()) != "task2_genai" and os.path.isdir("task2_genai"):
    os.chdir("task2_genai")
sys.path.insert(0, os.getcwd())
print("cwd:", os.getcwd(), "| Colab:", IN_COLAB)

In [ ]:
import json, logging, warnings, time
warnings.filterwarnings("ignore")
logging.basicConfig(level=logging.INFO, format="%(levelname)s %(name)s: %(message)s", force=True)
for n in ("httpx", "urllib3", "sentence_transformers", "chromadb", "huggingface_hub", "datasets"):
    logging.getLogger(n).setLevel(logging.WARNING)
import torch, transformers, peft, bitsandbytes, pandas as pd
from IPython.display import display, Markdown, Image
pd.set_option("display.max_colwidth", 140); pd.set_option("display.width", 220)
import config
assert torch.cuda.is_available(), "Runtime -> Change runtime type -> T4 GPU"
p = torch.cuda.get_device_properties(0)
print(f"GPU: {p.name} | {p.total_memory/2**30:.1f} GiB | compute capability {p.major}.{p.minor}")
print("torch", torch.__version__, "| transformers", transformers.__version__, "| peft", peft.__version__,
      "| bitsandbytes", bitsandbytes.__version__)

In [ ]:
# HF token from Colab Secrets (key icon) - never printed, never committed.
from src.llm import read_secret
HF_TOKEN = read_secret("HF_TOKEN")
print("HF token loaded:", bool(HF_TOKEN), "| target repo:", config.HF_REPO_ID)

WANDB_KEY = read_secret("WANDB_API_KEY")
REPORT_TO = "none"
if WANDB_KEY:
    import subprocess; subprocess.run([sys.executable, "-m", "pip", "install", "-q", "wandb"])
    import wandb; wandb.login(key=WANDB_KEY); REPORT_TO = "wandb"
print("report_to:", REPORT_TO)

## 1. Data and sequence-length check

In [ ]:
from src.format_split import load_split
from src import train as T
tok = T.load_tokenizer()
print({s: len(load_split(s)) for s in ("train", "val", "test")})
print("Token lengths (prompt + target, after chat template):", T.token_length_stats(tok))

## 2. Hyperparameters and justification
Every value is set explicitly in `config.py` / `src/train.py`. None is left at a library default without a reason.

| Parameter | Value | Justification |
|---|---|---|
| `load_in_4bit`, `bnb_4bit_quant_type` | True, **nf4** | QLoRA. NF4 is the information-theoretically optimal 4-bit code for normally distributed weights, so the 3.8B base fits in about 2.3 GB of VRAM. |
| `bnb_4bit_use_double_quant` | True | Quantises the quantisation constants too, saving about 0.37 bits per parameter (QLoRA paper). This is free headroom on a 15 GB T4. |
| `bnb_4bit_compute_dtype` / mixed precision | **float16** / `fp16=True` | The T4 (compute capability 7.5) has **no native bfloat16**. The blueprint's bf16 would error or fall back to slow emulation. fp16 with loss scaling is the correct choice on Turing GPUs. |
| `target_modules` | `qkv_proj, o_proj, gate_up_proj, down_proj` | All attention **and** MLP projections. Phi-3 **fuses** Q/K/V into `qkv_proj` and gate/up into `gate_up_proj`, so the blueprint's `q_proj/k_proj/...` names would match nothing in this model. Adapting the MLPs as well as attention clearly helps on new output formats (QLoRA paper: all-linear LoRA is needed to match full fine-tuning). |
| LoRA `r` | 16 | The task is narrow (fixed schema, 10 labels). r=8 risks under-fitting the label/risk decisions, while r=64 adds about 4x the parameters, which 192 examples cannot constrain (over-fitting risk). |
| `lora_alpha` | 32 | α/r = 2 sets the update scale. This widely used ratio keeps the effective learning rate of the adapter stable if r is changed. |
| `lora_dropout` | 0.05 | Light regularisation for a small dataset. Higher dropout slows convergence within 3 epochs. |
| `learning_rate` | 2e-4 | The standard QLoRA learning rate for 3–7B models. Adapters start at zero (B=0) and need a larger step than full fine-tuning (about 1e-5). |
| `lr_scheduler_type` | cosine | Smooth decay towards 0 over a short run gives a lower final loss than a constant rate and avoids a hard step change. |
| `warmup_ratio` | 0.1 | 4 warm-up steps out of 36 (192 examples / effective batch 16 = 12 steps per epoch × 3). Because there are so few steps, 0.03 would round to a single step. Warm-up protects the freshly initialised adapters and the fp16 loss scaler. |
| `num_train_epochs` | 3 | With 192 examples, 1 epoch is 12 optimiser steps, which is too few to learn the schema. More than 3 epochs over-fits 192 synthetic examples. Validation loss is checked every epoch and the **best epoch is kept** (`load_best_model_at_end`). |
| `per_device_train_batch_size` | 4 | Sequences are at most 600 tokens (mean about 400), so 4 fits easily on a T4 with gradient checkpointing (peak memory is printed below). |
| `gradient_accumulation_steps` | 4 | Effective batch 16 gives a smoother gradient and stable fp16 training, with 12 updates per epoch. |
| `max_seq_length` | 1024 | The longest example is 600 tokens (measured above), so **nothing is truncated**, and the limit leaves headroom for longer real clauses. Padding is dynamic, so the cap costs no memory. |
| `optim` | `paged_adamw_8bit` | The QLoRA optimiser. 8-bit Adam states plus paging to CPU absorb memory spikes during long sequences. |
| `max_grad_norm` | 0.3 | The QLoRA paper's clipping value. It stabilises fp16 updates of 4-bit LoRA. |
| `weight_decay` | 0.0 | The adapters are small and dropout already regularises them. Decaying LoRA weights towards 0 only shrinks the update. |
| `gradient_checkpointing` | True (non-reentrant) | Trades about 30% compute for a large activation-memory saving, which is the main protection against OOM on 15 GB. |
| loss masking | assistant tokens only | Loss is computed on the JSON answer plus `<|end|>`, not on the system/user prompt. The model learns to extract, not to repeat the prompt, and the loss values become interpretable. |
| `group_by_length` | True | Batches similar-length examples together to reduce padding waste. |
| eval / save strategy | per epoch, best by `eval_loss` | Produces the per-epoch validation curve the brief requires and guards against over-fitting. |
| `seed` / `data_seed` | 42 | Makes runs reproducible. |

In [ ]:
display(pd.DataFrame([{k: getattr(config, k) for k in [
    "BASE_MODEL", "LORA_R", "LORA_ALPHA", "LORA_DROPOUT", "LEARNING_RATE", "LR_SCHEDULER", "WARMUP_RATIO",
    "NUM_EPOCHS", "PER_DEVICE_BATCH", "GRAD_ACCUM", "MAX_SEQ_LENGTH", "OPTIMIZER", "MAX_GRAD_NORM",
    "WEIGHT_DECAY"]}]).T.rename(columns={0: "value"}))
print("LoRA targets:", config.LORA_TARGET_MODULES)

## 3. Load the 4-bit NF4 base model and attach LoRA adapters

In [ ]:
model = T.load_4bit_model()
print(model.config.quantization_config)
print(f"4-bit model memory footprint: {model.get_memory_footprint()/2**30:.2f} GiB")
model = T.prepare_peft_model(model)
model.print_trainable_parameters()

## 4. Train, logging train and validation loss per epoch

In [ ]:
ds = T.build_datasets(tok)
print(ds)
torch.cuda.reset_peak_memory_stats()
t0 = time.time()
trainer, cb, epoch0, result = T.train(model, tok, ds, output_dir="checkpoints", report_to=REPORT_TO)
print(f"Training time: {(time.time()-t0)/60:.1f} min | peak GPU memory: {torch.cuda.max_memory_allocated()/2**30:.2f} GiB")
print(result)

In [ ]:
table = T.loss_table(trainer, cb, epoch0)
display(table.style.format({"train_loss": "{:.4f}", "val_loss": "{:.4f}"}).hide(axis="index"))
v = table["val_loss"].tolist()
print("Validation loss by epoch:", [round(x, 4) for x in v])
print("Strictly decreasing across epochs:", all(b < a for a, b in zip(v, v[1:])))
print("Best checkpoint kept:", trainer.state.best_model_checkpoint)
config.OUTPUT_DIR.mkdir(exist_ok=True)
table.to_csv(config.OUTPUT_DIR / "loss_per_epoch.csv", index=False)
json.dump(trainer.state.log_history, open(config.OUTPUT_DIR / "trainer_log_history.json", "w"), indent=1)
fig = T.plot_losses(trainer, table, config.OUTPUT_DIR / "loss_curve.png")

In [ ]:
# Raw per-step log (evidence of manual logging)
display(pd.DataFrame(trainer.state.log_history).round(4))

### Memory / OOM notes
The memory budget was planned to avoid OOM on the 15 GB T4: a 4-bit base (about 2.3 GB),
LoRA adapters plus 8-bit paged optimiser states (under 0.5 GB), and activations kept small by
gradient checkpointing and dynamic padding of ≤ 600-token sequences. Peak usage is printed above.
If an OOM does occur on a smaller GPU, apply these fixes in order:
(1) `PER_DEVICE_BATCH=2, GRAD_ACCUM=8`, which keeps the effective batch at 16;
(2) `MAX_SEQ_LENGTH=768`;
(3) keep the paged optimiser and checkpointing enabled.
The merge step loads an fp16 base (about 7.6 GB), so the training model is deleted and the CUDA cache emptied first.

## 5. Save the adapter, merge into an fp16 base, and push to the Hugging Face Hub

In [ ]:
ADAPTER_DIR, MERGED_DIR = "adapter", "merged"
trainer.model.save_pretrained(ADAPTER_DIR)        # best epoch (load_best_model_at_end)
tok.save_pretrained(ADAPTER_DIR)
del trainer, model; T.free_gpu()
print(f"GPU memory after freeing: {torch.cuda.memory_allocated()/2**30:.2f} GiB")
merged = T.merge_adapter(ADAPTER_DIR, MERGED_DIR)   # PeftModel(...).merge_and_unload() on fp16 base
print(type(merged).__name__, "| merged params:", f"{sum(p.numel() for p in merged.parameters())/1e9:.2f}B")
print(sorted(os.listdir(MERGED_DIR)))

In [ ]:
# Sanity check: the merged model answers a validation clause in the target format.
from src.evaluate import generate_batch, build_prompt_messages
tok_l = T.load_tokenizer(MERGED_DIR)
row = load_split("val")[0]
out = generate_batch(merged, tok_l, [build_prompt_messages(row["messages"][1]["content"])])[0]
print("CLAUSE:", row["messages"][1]["content"]); print("MERGED MODEL:", out["raw"]); print("REFERENCE:", row["messages"][2]["content"])

In [ ]:
T.write_model_card(MERGED_DIR, table)
if HF_TOKEN:
    url = T.push(MERGED_DIR, token=HF_TOKEN)
    from huggingface_hub import HfApi
    api = HfApi(token=HF_TOKEN)
    for f in ("loss_per_epoch.csv", "loss_curve.png", "trainer_log_history.json"):
        api.upload_file(path_or_fileobj=str(config.OUTPUT_DIR / f), path_in_repo=f"training/{f}",
                        repo_id=config.HF_REPO_ID)
    print("Merged model:", url)
else:
    print("HF_TOKEN missing - model saved locally only in", MERGED_DIR)